# **Author**

*   Kumar Jayant


# **Notions of distance - Part 3**


Metrics covered:
* Haversine
* Jaccard
* Seorensen-Dice
* Hamming





Import required packages

In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from scipy.stats import norm, entropy
from scipy.stats import chi2
from sklearn.covariance import MinCovDet, EllipticEnvelope
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics.pairwise import haversine_distances
from sklearn.neighbors import BallTree
from sklearn.cluster import DBSCAN
from IPython.display import display_html


import warnings
# Suppress all warnings completely
warnings.filterwarnings('ignore')

For Haversine illustration, I created a sythetic dataset with 30 datapoints.

Datapoints being places with latitudes, and longitudes of places in Bangalore

In [2]:
# Load the data
file_path = "/content/bangalore_places_haversine.csv"
bang_df = pd.read_csv(file_path, encoding='ISO-8859-1')
bang_df

,id,place,area,category,latitude,longitude
0,1,Kempegowda International Airport,Devanahalli,Transport hub,13.1986,77.7066
1,2,KSR Bengaluru City Railway Station (Majestic),Majestic,Transport hub,12.9781,77.5697
2,3,Yeshwanthpur Railway Station,Yeshwanthpur,Transport hub,13.0230,77.5500
3,4,KR Puram Railway Station,KR Puram,Transport hub,13.0007,77.6763
4,5,Hebbal Flyover,Hebbal,Transport hub,13.0358,77.5970
5,6,Silk Board Junction,Silk Board,Transport hub,12.9172,77.6229
6,7,MG Road Metro Station,MG Road,Central business district,12.9755,77.6067
7,8,Vidhana Soudha,Ambedkar Veedhi,Central business district,12.9797,77.5907
8,9,Cubbon Park,Central Bengaluru,Leisure,12.9763,77.5929
9,10,Lalbagh Botanical Garden,Lalbagh,Leisure,12.9507,77.5848


### Variant 1: pairwise distance

In [3]:
# 1. Compute pairwise Haversine distances in km
coords_rad = np.radians(bang_df[["latitude", "longitude"]])
dist_matrix_km = haversine_distances(coords_rad) * 6371.0

In [4]:
# 2. Create clean, concise, unique names for rows and columns
place_names = bang_df["place"].tolist()

In [5]:
# Short name mapping dictionary for Bengaluru locations
short_names_map = {
    "Kempegowda International Airport": "Airport",
    "KSR Bengaluru City Railway Station (Majestic)": "Majestic",
    "Yeshwanthpur Railway Station": "Yeshwanthpur",
    "KR Puram Railway Station": "KR Puram",
    "Hebbal Flyover": "Hebbal",
    "Silk Board Junction": "Silk Board",
    "MG Road Metro Station": "MG Road",
    "Vidhana Soudha": "Vidhana Soudha",
    "Cubbon Park": "Cubbon Park",
    "Lalbagh Botanical Garden": "Lalbagh",
    "Bangalore Palace": "Blr Palace",
    "Bannerghatta National Park": "Bannerghatta",
    "Wonderla Amusement Park": "Wonderla",
    "Nandi Hills": "Nandi Hills",
    "Electronic City": "E-City",
    "ITPL": "ITPL",
    "Manyata Tech Park": "Manyata Park",
    "Sarjapur Road": "Sarjapur Rd",
    "Marathahalli": "Marathahalli",
    "Forum Mall": "Forum Mall",
    "100 Feet Road Koramangala": "100 Ft Rd",
    "4th Block Jayanagar": "4th Block",
    "HSR Layout": "HSR Layout",
    "BTM Layout": "BTM Layout",
    "18th Cross Malleshwaram": "18th Cross",
    "Rajajinagar": "Rajajinagar",
    "Banashankari": "Banashankari",
    "Yelahanka New Town": "Yelahanka",
    "Kengeri Satellite Town": "Kengeri",
    "Indian Institute of Science (IISc)": "IISc",
}

In [6]:
# Apply mapping cleanly using .get() to prevent missing/duplicate key errors
clean_labels = [short_names_map.get(name, name) for name in place_names]

# Create DataFrame with distinct index & column labels
distance_df = pd.DataFrame(
    dist_matrix_km, index=clean_labels, columns=clean_labels
)

In [7]:
# 3. Apply Pandas Styler safely (now that index and columns are strictly unique)
styled_matrix = (
    distance_df.style.background_gradient(
        cmap="YlOrRd", axis=None
    )  # Heatmap color gradient
    .format("{:.1f}")  # Format distance to 1 decimal place
    .set_properties(
        **{
            "text-align": "center",
            "font-size": "11px",
            "padding": "6px",
            "min-width": "60px",
            "max-width": "75px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1e293b"),
                    ("color", "#ffffff"),
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                ],
            }
        ]
    )
)

# Display in Jupyter
styled_matrix

,Airport,Majestic,Yeshwanthpur,KR Puram,Hebbal,Silk Board,MG Road,Vidhana Soudha,Cubbon Park,Lalbagh,Blr Palace,Bannerghatta Biological Park,Wonderla,Nandi Hills,Electronic City (Infosys campus),ITPL,Manyata Park,Sarjapur Road (Wipro campus),Marathahalli Bridge,Forum Mall,100 Feet Road,4th Block,HSR Layout,BTM Layout,18th Cross,Rajajinagar,Banashankari Temple,Yelahanka,Kengeri,Indian Institute of Science
Airport,0.0,28.7,25.9,22.2,21.6,32.6,27.1,27.4,27.6,30.6,25.4,46.5,52.3,19.2,39.6,23.8,19.3,32.2,26.9,31.1,26.2,32.8,32.6,33.1,26.2,28.5,34.6,16.2,40.1,24.8
Majestic,28.7,0.0,5.4,11.8,7.1,8.9,4.0,2.3,2.5,3.5,3.3,19.8,24.3,45.3,17.7,18.1,9.4,14.5,14.4,6.6,7.8,6.5,11.2,8.1,2.8,2.4,7.0,13.9,11.9,4.9
Yeshwanthpur,25.9,5.4,0.0,13.9,5.3,14.2,8.1,6.5,7.0,8.9,5.3,25.0,26.5,41.2,23.1,20.6,8.2,19.1,17.9,11.9,11.4,11.9,16.3,13.5,3.1,3.5,12.2,10.0,14.4,1.9
KR Puram,22.2,11.8,13.9,0.0,9.4,10.9,8.0,9.6,9.4,11.4,9.1,24.8,35.1,41.1,17.3,6.7,7.8,10.1,5.6,10.2,5.0,12.3,10.4,11.8,11.4,13.5,14.6,14.1,23.1,12.1
Hebbal,21.6,7.1,5.3,9.4,0.0,13.5,6.8,6.3,6.6,9.6,4.1,26.3,30.9,38.3,22.2,16.1,2.9,16.8,14.3,11.4,8.6,12.3,14.9,13.3,4.6,6.9,13.6,7.2,18.5,3.6
Silk Board,32.6,8.9,14.2,10.9,13.5,0.0,6.7,7.8,7.3,5.6,9.7,13.9,25.8,50.8,8.9,14.5,14.4,6.6,9.6,2.3,6.4,3.3,2.7,1.4,11.1,11.3,5.3,20.6,15.0,13.1
MG Road,27.1,4.0,8.1,8.0,6.8,6.7,0.0,1.8,1.5,3.6,3.0,19.8,27.3,44.7,15.6,14.1,8.0,11.0,10.4,4.6,3.8,5.8,8.4,6.6,5.0,6.2,7.6,14.0,15.1,6.7
Vidhana Soudha,27.4,2.3,6.5,9.6,6.3,7.8,1.8,0.0,0.4,3.3,2.1,20.0,26.2,44.6,16.7,15.8,8.1,12.7,12.2,5.5,5.5,6.1,9.8,7.3,3.4,4.4,7.4,13.5,13.8,5.3
Cubbon Park,27.6,2.5,7.0,9.4,6.6,7.3,1.5,0.4,0.0,3.0,2.5,19.7,26.1,44.9,16.3,15.6,8.4,12.3,11.9,5.0,5.3,5.7,9.3,6.9,3.8,4.7,7.1,13.8,13.8,5.8
Lalbagh,30.6,3.5,8.9,11.4,9.6,5.6,3.6,3.3,3.0,0.0,5.4,16.8,23.8,47.9,14.3,16.9,11.4,11.6,12.6,3.4,6.6,3.0,8.1,4.7,6.1,5.7,4.1,16.7,11.7,8.1


### Variant 2: distance between 2 selected points

In [9]:
place_a = bang_df[
    bang_df["place"] == "Kempegowda International Airport"
].iloc[0]

place_b = bang_df[
    bang_df["place"] == "KSR Bengaluru City Railway Station (Majestic)"
].iloc[0]

# Convert lat/lon to radians
coords_a = np.radians([place_a["latitude"], place_a["longitude"]])
coords_b = np.radians([place_b["latitude"], place_b["longitude"]])

# Compute distance
dist_km = haversine_distances([coords_a], [coords_b])[0][0] * 6371.0

print(f"Distance: {dist_km:.2f} km")

Distance: 28.65 km


### Variant 3: K nearest neighbours

For the ilustration K = 2 in the snippet

In [10]:
# 1. Earth's mean radius in kilometers
R = 6371.0

# 2. Extract latitude & longitude and convert degrees to RADIANS
X = np.radians(bang_df[["latitude", "longitude"]].to_numpy())

# 3. Fit BallTree using Haversine metric
tree = BallTree(X, metric="haversine")

# --- QUERY 1: k-Nearest Neighbors ---
# Find the 2 nearest neighbors to index 0
k=2
dist_rad, idx = tree.query(X[[0]], k=2)

# Convert return distances from radians back to kilometers
dist_km = dist_rad * R

print(f"Nearest {k} Neighbors Result for {bang_df.iloc[0]['place']}:")
for d, i in zip(dist_km[0], idx[0]):
    place_name = bang_df.iloc[i]["place"]
    print(f" - {place_name} (Index {i}): {d:.2f} km")

Nearest 2 Neighbors Result for Kempegowda International Airport:
 - Kempegowda International Airport (Index 0): 0.00 km
 - Yelahanka New Town (Index 27): 16.16 km


### Variant 4: Nearest neighbours within X kms radius

For the illustration X = 20 Kms

Method 1

In [11]:
radius_km = 20.0
radius_rad = radius_km / R  # Convert km to radians

inside_indices = tree.query_radius(X[[0]], r=radius_rad)[0]

print(f"\nPlaces within {radius_km} km of '{bang_df.iloc[0]['place']}':")
for i in inside_indices:
    place_name = bang_df.iloc[i]["place"]
    print(f" - {place_name} (Index {i})")


Places within 20.0 km of 'Kempegowda International Airport':
 - Kempegowda International Airport (Index 0)
 - Nandi Hills (Index 13)
 - Manyata Tech Park (Index 16)
 - Yelahanka New Town (Index 27)


### Variant 5: Nearest neighbours within X kms radius

Working of DBSCAN

*   min_samples = number of points to be called a cluster
*   eps (in radians) = radius in which the min_samples have to fall


For the snippet below, If there are 3 points which fall which fall within radius of 1.5 Km they will be assigned a cluter ID starting 0.

For all the points which would not fall in a cluster shall fall in cluster ID -1

In [15]:
# 1. Earth radius constant in kilometers
R = 6371.0

# 2. Convert latitude and longitude from degrees to RADIANS
X = np.radians(bang_df[["latitude", "longitude"]].to_numpy())

# 3. Configure DBSCAN: 1.5 km radius, minimum 10 points per cluster
db = DBSCAN(
    eps=1.5 / R,  # 1.5 km converted to radians
    min_samples=3,  # Minimum core density threshold
    metric="haversine",  # Great-circle metric
    algorithm="ball_tree",  # Fast spatial tree lookups
)

# 4. Predict cluster labels
bang_df["cluster_label"] = db.fit_predict(X)

# View cluster distribution breakdown
print(bang_df["cluster_label"].value_counts())

cluster_label
-1    27
 0     3
Name: count, dtype: int64


For our example: 3 points out of 30 fall in a cluster, balance do not have a cluster.

The clustered points have been printed below.

In [16]:
# places where cluster label == 0
bang_df[bang_df['cluster_label'] == 0]

,id,place,area,category,latitude,longitude,cluster_label
6,7,MG Road Metro Station,MG Road,Central business district,12.9755,77.6067,0
7,8,Vidhana Soudha,Ambedkar Veedhi,Central business district,12.9797,77.5907,0
8,9,Cubbon Park,Central Bengaluru,Leisure,12.9763,77.5929,0


**Jaccard, Sorensen-Dice, and Hamming distance/similarity starts here**

##### Step 1: Data Preprocessing to ready the data for analysis

1. Ingest the ola dataset.
2. Data profiling
3. Data cleansing (dropping features not needed for analysis)
4. Data imputation
5. Short Description of the final features for familiarity with the dataset


Ingesting the dataset

In [7]:
# Load the data
file_path = "/content/ola.csv"
ola_df = pd.read_csv(file_path, encoding='ISO-8859-1')
ola_df.head()

,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Booking Value,Payment Method,Ride Distance,Driver Ratings,Customer Rating
0,329258,Auto,Area-3,Area-2,5.42,18.46,868.06,Wallet,28.5,4.4,4.4
1,201414,Mini,Area-7,Area-6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,301629,Bike,Area-40,Area-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,319684,Prime Sedan,Area-11,Area-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,330283,Bike,Area-41,Area-45,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
ola_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49999 entries, 0 to 49998
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer ID      49999 non-null  int64  
 1   Vehicle Type     49999 non-null  object 
 2   Pickup Location  49999 non-null  object 
 3   Drop Location    49999 non-null  object 
 4   Avg VTAT         33484 non-null  float64
 5   Avg CTAT         33484 non-null  float64
 6   Booking Value    33484 non-null  float64
 7   Payment Method   33484 non-null  object 
 8   Ride Distance    33484 non-null  float64
 9   Driver Ratings   33484 non-null  float64
 10  Customer Rating  33484 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 4.2+ MB


In [9]:
# convert Customer ID to object datatype
ola_df['Customer ID'] = ola_df['Customer ID'].astype(str)

In [10]:
ola_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49999 entries, 0 to 49998
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer ID      49999 non-null  object 
 1   Vehicle Type     49999 non-null  object 
 2   Pickup Location  49999 non-null  object 
 3   Drop Location    49999 non-null  object 
 4   Avg VTAT         33484 non-null  float64
 5   Avg CTAT         33484 non-null  float64
 6   Booking Value    33484 non-null  float64
 7   Payment Method   33484 non-null  object 
 8   Ride Distance    33484 non-null  float64
 9   Driver Ratings   33484 non-null  float64
 10  Customer Rating  33484 non-null  float64
dtypes: float64(6), object(5)
memory usage: 4.2+ MB


**Data Imputation**

The following fields will be imputed for missing data.

**Numerical fields**

1. Avg VTAT
2. Avg CTAT
3. Booking Value
4. Ride Distance
5. Driver Ratings
6. Customer Rating

Logic for imputation - If there are multiple customer IDs for these fields then average of the customer IDs will be used for imputation else the previous row data shall be rolled over.

**Categorical field**

1. Payment Method

Logic for imputation - If there are multiple customer IDs for this field then max instance of the customer IDs will be used for imputation else the previous row data shall be rolled over.

In [11]:
# Imputing numerical features

ola_df['Avg VTAT'] = ola_df['Avg VTAT'].fillna(ola_df.groupby('Customer ID')['Avg VTAT'].transform('mean'))
ola_df['Avg VTAT'] = ola_df['Avg VTAT'].ffill().bfill()

ola_df['Avg CTAT'] = ola_df['Avg CTAT'].fillna(ola_df.groupby('Customer ID')['Avg CTAT'].transform('mean'))
ola_df['Avg CTAT'] = ola_df['Avg CTAT'].ffill().bfill()

ola_df['Booking Value'] = ola_df['Booking Value'].fillna(ola_df.groupby('Customer ID')['Booking Value'].transform('mean'))
ola_df['Booking Value'] = ola_df['Booking Value'].ffill().bfill()

ola_df['Ride Distance'] = ola_df['Ride Distance'].fillna(ola_df.groupby('Customer ID')['Ride Distance'].transform('mean'))
ola_df['Ride Distance'] = ola_df['Ride Distance'].ffill().bfill()

ola_df['Driver Ratings'] = ola_df['Driver Ratings'].fillna(ola_df.groupby('Customer ID')['Driver Ratings'].transform('mean'))
ola_df['Driver Ratings'] = ola_df['Driver Ratings'].ffill().bfill()

ola_df['Customer Rating'] = ola_df['Customer Rating'].fillna(ola_df.groupby('Customer ID')['Customer Rating'].transform('mean'))
ola_df['Customer Rating'] = ola_df['Customer Rating'].ffill().bfill()

In [12]:
# Imputing categorical feature

# Imputing categorical feature safely
ola_df['Payment Method'] = ola_df['Payment Method'].fillna(
    ola_df.groupby('Customer ID')['Payment Method'].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
)
ola_df['Payment Method'] = ola_df['Payment Method'].ffill().bfill()

In [13]:
ola_df.head()

,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Booking Value,Payment Method,Ride Distance,Driver Ratings,Customer Rating
0,329258,Auto,Area-3,Area-2,5.42,18.46,868.06,Wallet,28.5,4.4,4.4
1,201414,Mini,Area-7,Area-6,5.42,18.46,868.06,Wallet,28.5,4.4,4.4
2,301629,Bike,Area-40,Area-24,5.42,18.46,868.06,Wallet,28.5,4.4,4.4
3,319684,Prime Sedan,Area-11,Area-24,5.42,18.46,868.06,Wallet,28.5,4.4,4.4
4,330283,Bike,Area-41,Area-45,5.42,18.46,868.06,Wallet,28.5,4.4,4.4


In [14]:
# Isolate categorical features (Vehicle, type and payment method) from ola dataframe

ola_cat = []
cat_cols = ['Customer ID', 'Vehicle Type', 'Payment Method']
ola_cat = ola_df[cat_cols].copy()
ola_cat.head()

,Customer ID,Vehicle Type,Payment Method
0,329258,Auto,Wallet
1,201414,Mini,Wallet
2,301629,Bike,Wallet
3,319684,Prime Sedan,Wallet
4,330283,Bike,Wallet


In [15]:
# Pick 10 customers with at least 2 completed rides
counts = ola_cat["Customer ID"].value_counts()
ids = counts[counts >= 2].index.to_series().sample(10, random_state=42)
sample = ola_cat[ola_cat["Customer ID"].isin(ids)]
sample


,Customer ID,Vehicle Type,Payment Method
997,544045,eBike,Wallet
1479,544045,eBike,Wallet
8943,919857,Auto,Card
9753,964477,Prime SUV,Card
10268,866195,Bike,Card
13059,866195,Mini,Card
14326,295818,eBike,Cash
15407,802774,Prime SUV,UPI
17305,919857,Bike,Card
17806,389033,Auto,Card


In [16]:
# Pivot: one row per customer, vehicle types then payment methods, 1 = used at least once
veh = pd.crosstab(sample["Customer ID"], sample["Vehicle Type"])
pay = pd.crosstab(sample["Customer ID"], sample["Payment Method"])
pivot = (pd.concat([veh, pay], axis=1) > 0).astype(int)
pivot

,Auto,Bike,Mini,Prime Plus,Prime SUV,Prime Sedan,eBike,Card,Cash,UPI,Wallet
Customer ID,,,,,,,,,,,
175986,0,0,0,1,0,0,1,0,0,1,0
295818,0,0,1,0,0,0,1,0,1,1,0
389033,1,0,0,0,1,0,0,1,0,1,0
476821,1,0,0,0,1,0,0,1,0,0,1
544045,0,0,0,0,0,0,1,0,0,0,1
602613,0,0,1,0,0,0,1,1,0,1,0
802774,0,0,1,0,1,0,0,0,1,1,0
866195,0,1,1,0,0,0,0,1,0,0,0
919857,1,1,0,0,0,0,0,1,0,0,0


In [22]:
# Distance matrices
X = pivot.values.astype(bool)
jaccard_distance = pairwise_distances(X, metric="jaccard")
jaccard_similarity = 1 - jaccard_distance
dice_distance    = pairwise_distances(X, metric="dice")
dice_similarity  = 1 - dice_distance
hamming_distance = pairwise_distances(X, metric="hamming")
hamming_similarity = 1 - hamming_distance

In [24]:
# Create DataFrame with distinct index & column labels for all the 3 metrics
distance_jaccard_df = pd.DataFrame(
    jaccard_distance, index=pivot.index, columns=pivot.index
)

similarity_jaccard_df = pd.DataFrame(
    jaccard_similarity, index=pivot.index, columns=pivot.index
)

distance_dice_df = pd.DataFrame(
    dice_distance, index=pivot.index, columns=pivot.index
)

similarity_dice_df = pd.DataFrame(
    dice_similarity, index=pivot.index, columns=pivot.index
)

distance_hamming_df = pd.DataFrame(
    hamming_distance, index=pivot.index, columns=pivot.index
)

similarity_hamming_df = pd.DataFrame(
    hamming_similarity, index=pivot.index, columns=pivot.index
)



In [25]:
def side_by_side(*dfs, titles=()):
    html = ""
    for df, title in zip(dfs, titles):
        html += (df.round(2).style
                   .set_caption(title)
                   .set_table_attributes('style="display:inline-block; margin-right:30px; vertical-align:top"')
                   .to_html())
    display_html(html, raw=True)

side_by_side(distance_jaccard_df, similarity_jaccard_df,
             titles=("Jaccard distance", "Jaccard similarity"))

Customer ID,175986,295818,389033,476821,544045,602613,802774,866195,919857,964477
Customer ID,,,,,,,,,,
175986,0.000000,0.600000,0.830000,1.000000,0.750000,0.600000,0.830000,1.000000,1.000000,1.000000
295818,0.600000,0.000000,0.860000,1.000000,0.800000,0.400000,0.400000,0.830000,1.000000,1.000000
389033,0.830000,0.860000,0.000000,0.400000,1.000000,0.670000,0.670000,0.830000,0.600000,0.600000
476821,1.000000,1.000000,0.400000,0.000000,0.800000,0.860000,0.860000,0.830000,0.600000,0.600000
544045,0.750000,0.800000,1.000000,0.800000,0.000000,0.800000,1.000000,1.000000,1.000000,1.000000
602613,0.600000,0.400000,0.670000,0.860000,0.800000,0.000000,0.670000,0.600000,0.830000,0.830000
802774,0.830000,0.400000,0.670000,0.860000,1.000000,0.670000,0.000000,0.830000,1.000000,0.830000
866195,1.000000,0.830000,0.830000,0.830000,1.000000,0.600000,0.830000,0.000000,0.500000,0.800000
919857,1.000000,1.000000,0.600000,0.600000,1.000000,0.830000,1.000000,0.500000,0.000000,0.800000


In [26]:
def side_by_side(*dfs, titles=()):
    html = ""
    for df, title in zip(dfs, titles):
        html += (df.round(2).style
                   .set_caption(title)
                   .set_table_attributes('style="display:inline-block; margin-right:30px; vertical-align:top"')
                   .to_html())
    display_html(html, raw=True)

side_by_side(distance_dice_df, similarity_dice_df,
             titles=("Dice distance", "Dice similarity"))

Customer ID,175986,295818,389033,476821,544045,602613,802774,866195,919857,964477
Customer ID,,,,,,,,,,
175986,0.000000,0.430000,0.710000,1.000000,0.600000,0.430000,0.710000,1.000000,1.000000,1.000000
295818,0.430000,0.000000,0.750000,1.000000,0.670000,0.250000,0.250000,0.710000,1.000000,1.000000
389033,0.710000,0.750000,0.000000,0.250000,1.000000,0.500000,0.500000,0.710000,0.430000,0.430000
476821,1.000000,1.000000,0.250000,0.000000,0.670000,0.750000,0.750000,0.710000,0.430000,0.430000
544045,0.600000,0.670000,1.000000,0.670000,0.000000,0.670000,1.000000,1.000000,1.000000,1.000000
602613,0.430000,0.250000,0.500000,0.750000,0.670000,0.000000,0.500000,0.430000,0.710000,0.710000
802774,0.710000,0.250000,0.500000,0.750000,1.000000,0.500000,0.000000,0.710000,1.000000,0.710000
866195,1.000000,0.710000,0.710000,0.710000,1.000000,0.430000,0.710000,0.000000,0.330000,0.670000
919857,1.000000,1.000000,0.430000,0.430000,1.000000,0.710000,1.000000,0.330000,0.000000,0.670000


In [27]:
def side_by_side(*dfs, titles=()):
    html = ""
    for df, title in zip(dfs, titles):
        html += (df.round(2).style
                   .set_caption(title)
                   .set_table_attributes('style="display:inline-block; margin-right:30px; vertical-align:top"')
                   .to_html())
    display_html(html, raw=True)

side_by_side(distance_hamming_df, similarity_hamming_df,
             titles=("Hamming distance", "Hamming similarity"))

Customer ID,175986,295818,389033,476821,544045,602613,802774,866195,919857,964477
Customer ID,,,,,,,,,,
175986,0.000000,0.270000,0.450000,0.640000,0.270000,0.270000,0.450000,0.550000,0.550000,0.550000
295818,0.270000,0.000000,0.550000,0.730000,0.360000,0.180000,0.180000,0.450000,0.640000,0.640000
389033,0.450000,0.550000,0.000000,0.180000,0.550000,0.360000,0.360000,0.450000,0.270000,0.270000
476821,0.640000,0.730000,0.180000,0.000000,0.360000,0.550000,0.550000,0.450000,0.270000,0.270000
544045,0.270000,0.360000,0.550000,0.360000,0.000000,0.360000,0.550000,0.450000,0.450000,0.450000
602613,0.270000,0.180000,0.360000,0.550000,0.360000,0.000000,0.360000,0.270000,0.450000,0.450000
802774,0.450000,0.180000,0.360000,0.550000,0.550000,0.360000,0.000000,0.450000,0.640000,0.450000
866195,0.550000,0.450000,0.450000,0.450000,0.450000,0.270000,0.450000,0.000000,0.180000,0.360000
919857,0.550000,0.640000,0.270000,0.270000,0.450000,0.450000,0.640000,0.180000,0.000000,0.360000


# End of Part 3